# RAG Validation Metrics

This notebook checks whether a PDF question-answering system finds useful information and gives a good answer. It is written as a small demonstration for a general audience.

We will ask five questions about the sample company policies, compare the answers with known reference answers, and score the results from **0 (poor) to 1 (strong)**.

**You need:** the `sample_pdfs` folder beside this notebook and an OpenAI API key in a `.env` file. Run the cells from top to bottom.

## 1. Install the needed packages

Run this once in the notebook. If asked, restart the notebook kernel.

In [ ]:
#%pip install -q langchain langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf python-dotenv pandas

## 2. Connect to OpenAI

The API key stays in your local `.env` file. Do not paste a real key into a notebook you plan to share.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file beside this notebook.")

chat_model = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
embedding_model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

## 3. Load the sample PDFs and make searchable pieces

RAG splits long documents into smaller pieces so it can find the parts related to a question.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_files = sorted(Path("sample_pdfs").glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError("Put the sample_pdfs folder beside this notebook.")

pages = []
for file in pdf_files:
    pages.extend(PyPDFLoader(str(file)).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print(f"Loaded {len(pdf_files)} PDFs and prepared {len(chunks)} searchable pieces.")

## 4. Build a simple RAG question-answer helper

For each question, the helper finds three relevant pieces of text, gives them to the AI model, and asks it to answer using only those pieces.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=embedding_model)
knowledge_base = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="validation_metrics_demo",
)
retriever = knowledge_base.as_retriever(search_kwargs={"k": 3})
chat_model_client = ChatOpenAI(model=chat_model, temperature=0)

def answer_question(question):
    found_pieces = retriever.invoke(question)
    context = "\n\n".join(piece.page_content for piece in found_pieces)
    prompt = f"""Answer using only the information below. If it is not there, say you could not find it.

Information:
{context}

Question: {question}
Answer:"""
    answer = chat_model_client.invoke(prompt).content
    return answer, context

answer, context = answer_question("How many paid annual leave days do full-time employees get?")
print(answer)

## 5. The five validation metrics

The AI evaluator will give each metric a score from **0 to 1**. Think of the metrics as five questions:

| Metric | In everyday language |
| --- | --- |
| Context precision | Were the pieces found mostly useful? |
| Context recall | Did the pieces include the information needed? |
| Faithfulness | Did the answer stick to what the documents say? |
| Answer relevance | Did the answer address the question? |
| Answer correctness | Does the answer agree with the reference answer? |

The scores are estimates from an AI evaluator. They are useful for a demo and for comparing changes, but a person should review important results.

## 6. Prepare example questions and reference answers

A reference answer is the answer we expect to find in the sample policy documents.

In [ ]:
questions = [
    ("How many paid annual leave days do full-time employees receive?",
     "They receive 18 days of paid annual leave per calendar year."),
    ("When should employees submit expense claims?",
     "Within 30 days of the purchase, with itemized receipts and the required business details."),
    ("What should an employee do after receiving a suspected phishing message?",
     "Do not click links or open attachments; report it to the IT help desk using the approved reporting method."),
    ("How many days per week may an employee work remotely?",
     "Up to two days per week, with manager approval."),
    ("What approval is needed for a single expense above 10,000 rupees?",
     "Manager approval is required."),
]

## 7. Score the answers

For each question, the notebook gets an answer and asks an AI evaluator to score it against the retrieved text and reference answer. It then shows the scores in a table.

In [ ]:
import json
import pandas as pd
from langchain_openai import ChatOpenAI

judge = ChatOpenAI(model=chat_model, temperature=0)
rows = []

for question, reference in questions:
    answer, context = answer_question(question)
    judge_prompt = f"""Evaluate this RAG answer. Give each score from 0 to 1.
Return only one JSON object with these keys: context_precision, context_recall, faithfulness, answer_relevance, answer_correctness.

context_precision: Were the retrieved text pieces relevant?
context_recall: Did the retrieved text contain the needed information?
faithfulness: Is the answer supported by the retrieved text?
answer_relevance: Does the answer address the question?
answer_correctness: Does the answer agree with the reference?

Question: {question}
Reference answer: {reference}
Retrieved text: {context}
Generated answer: {answer}
"""
    scores = json.loads(judge.invoke(judge_prompt).content)
    rows.append({
        "Question": question,
        "Generated answer": answer,
        "Precision": scores["context_precision"],
        "Recall": scores["context_recall"],
        "Faithfulness": scores["faithfulness"],
        "Relevance": scores["answer_relevance"],
        "Correctness": scores["answer_correctness"],
    })

results = pd.DataFrame(rows)
results.round(2)

## 8. Look at the average scores

The average gives a quick view across all five questions. If one score is low, inspect the questions and answers in the table to see what went wrong.

In [ ]:
score_columns = ["Precision", "Recall", "Faithfulness", "Relevance", "Correctness"]
results[score_columns].mean().round(2).to_frame("Average score (0–1)")

## How to explain the results

- **Low precision:** the search brought back too much unrelated text.
- **Low recall:** the useful detail may not have been found.
- **Low faithfulness:** the answer may include something the documents do not support.
- **Low relevance:** the answer may not directly answer the question.
- **Low correctness:** the answer may differ from the expected answer.

A low score is a clue to investigate, not a diagnosis by itself. The AI evaluator can make mistakes, so compare the score with the source documents.

## Summary

We tested five questions and used five simple metrics to check what RAG found and how well it answered. The scores help us spot areas to improve, while the documents and human review remain the final check.